In [15]:
# ============================================================
# PHASE 5 — REPEATABLE ETL PIPELINE
# ============================================================
#
# PURPOSE
# -------
# Build the validated working-capital dataset independently
# from the original dirty CSV.
#
# Phase 4 is NOT used as the transformation source.
# Phase 4 is used ONLY as a regression reference.
#
# Pipeline:
#
#   RAW CSV
#      ↓
#   EXTRACT
#      ↓
#   TRANSFORM / CLEAN
#      ↓
#   INTERNAL VALIDATION
#      ↓
#   SAVE PHASE 5 OUTPUT
#      ↓
#   REGRESSION AGAINST PHASE 4
#      ↓
#   REPORTS
#
# Important:
# - Raw input is never modified.
# - Exact duplicates are removed.
# - Raw invoice values are preserved before coercion.
# - Missing and invalid invoice amounts remain distinguishable.
# - Phase 4 business-rule definitions are reproduced.
# - Regression compares records by invoice_id, not row position.
# - Regression failures are reported rather than crashing the pipeline.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime
import hashlib


# ============================================================
# 2. FILE PATHS
# ============================================================

RAW_PATH = Path(
    r"C:\Users\hp\Downloads\Working_capital_dirty .csv"
)

PHASE4_PATH = Path(
    r"C:\Users\hp\Downloads\Working_capital_validated.csv"
)

OUTPUT_DIR = Path(
    r"C:\Users\hp\Downloads\Working_Capital_ETL"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Main output
OUTPUT_PATH = (
    OUTPUT_DIR
    / "Working_capital_ETL_output.csv"
)

# Reports
VALIDATION_REPORT_PATH = (
    OUTPUT_DIR
    / "Phase5_Validation_Report.csv"
)

REGRESSION_REPORT_PATH = (
    OUTPUT_DIR
    / "Phase5_Regression_Report.csv"
)

REGRESSION_DETAIL_PATH = (
    OUTPUT_DIR
    / "Phase5_Regression_Detail.csv"
)

EXCEPTION_REPORT_PATH = (
    OUTPUT_DIR
    / "Phase5_Exception_Report.csv"
)

QUALITY_REPORT_PATH = (
    OUTPUT_DIR
    / "Phase5_Quality_Report.csv"
)

RUN_LOG_PATH = (
    OUTPUT_DIR
    / "Phase5_Run_Log.csv"
)

SHA256_REPORT_PATH = (
    OUTPUT_DIR
    / "Phase5_SHA256_Report.csv"
)


# ============================================================
# 3. RUN METADATA
# ============================================================

RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RUN_TIMESTAMP = datetime.now().isoformat(
    timespec="seconds"
)


# ============================================================
# 4. EXPECTED RAW COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "invoice_date",
    "due_date",
    "payment_date",
    "invoice_amount",
    "payment_amount",
    "currency",
    "payment_method",
    "status",
]


# ============================================================
# 5. HELPER — STANDARDIZE COLUMN NAMES
# ============================================================

def standardize_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(
            " ",
            "_",
            regex=False
        )
    )

    return df


# ============================================================
# 6. HELPER — SAFE BOOLEAN
# ============================================================

def safe_bool(series):
    """
    Convert a nullable boolean Series into ordinary
    True/False values for NumPy operations.
    """

    return (
        series
        .fillna(False)
        .to_numpy(dtype=bool)
    )


# ============================================================
# 7. EXTRACT
# ============================================================

def extract_data():

    if not RAW_PATH.exists():
        raise FileNotFoundError(
            f"Raw input file not found:\n{RAW_PATH}"
        )

    df = pd.read_csv(
        RAW_PATH
    )

    df = standardize_column_names(
        df
    )

    missing_columns = [
        col
        for col in EXPECTED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing expected columns: "
            f"{missing_columns}"
        )

    return df


# ============================================================
# 8. TRANSFORM / CLEAN
# ============================================================

def transform_data(df):

    df = df.copy()


    # --------------------------------------------------------
    # 8.1 Preserve raw values BEFORE any transformation
    # --------------------------------------------------------

    df["_raw_invoice_amount"] = (
        df["invoice_amount"]
        .astype("string")
        .str.strip()
    )

    df["_raw_invoice_date"] = (
        df["invoice_date"]
        .astype("string")
        .str.strip()
    )

    df["_raw_due_date"] = (
        df["due_date"]
        .astype("string")
        .str.strip()
    )


    # --------------------------------------------------------
    # 8.2 Identify genuinely missing invoice amounts
    # --------------------------------------------------------

    df["_invoice_amount_is_missing"] = (
        df["_raw_invoice_amount"].isna()
        | df["_raw_invoice_amount"].eq("")
    )


    # --------------------------------------------------------
    # 8.3 Parse invoice amount
    # --------------------------------------------------------

    invoice_amount_for_parsing = (
        df["_raw_invoice_amount"]
        .str.replace(
            ",",
            "",
            regex=False
        )
    )

    invoice_amount_for_parsing = (
        invoice_amount_for_parsing.mask(
            invoice_amount_for_parsing
            .str.lower()
            .isin(
                [
                    "not available",
                    "unknown",
                    "-",
                    "n/a",
                    "na",
                ]
            )
        )
    )

    df["_invoice_amount_numeric"] = (
        pd.to_numeric(
            invoice_amount_for_parsing,
            errors="coerce"
        )
    )


    # --------------------------------------------------------
    # 8.4 Invoice amount classification
    #
    # IMPORTANT:
    # This reproduces Phase 4 exactly.
    # --------------------------------------------------------

    df["flag_missing_invoice_amount"] = (
        df["_invoice_amount_is_missing"]
    ).astype("boolean")


    df["flag_invalid_invoice_amount"] = (
        ~df["_invoice_amount_is_missing"]
        & df["_invoice_amount_numeric"].isna()
    ).astype("boolean")


    df["flag_unusable_invoice_amount"] = (
        df["flag_missing_invoice_amount"]
        | df["flag_invalid_invoice_amount"]
    ).astype("boolean")


    # --------------------------------------------------------
    # 8.5 Replace invoice amount with clean numeric value
    # --------------------------------------------------------

    df["invoice_amount"] = (
        df["_invoice_amount_numeric"]
    )


    # --------------------------------------------------------
    # 8.6 Parse payment amount
    # --------------------------------------------------------

    df["payment_amount"] = pd.to_numeric(
        df["payment_amount"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # 8.7 Parse dates
    # --------------------------------------------------------

    df["invoice_date"] = pd.to_datetime(
        df["invoice_date"],
        errors="coerce"
    )

    df["due_date"] = pd.to_datetime(
        df["due_date"],
        errors="coerce"
    )

    df["payment_date"] = pd.to_datetime(
        df["payment_date"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # 8.8 Date invalidity flags
    # --------------------------------------------------------

    invoice_date_was_present = (
        df["_raw_invoice_date"].notna()
        & df["_raw_invoice_date"].ne("")
    )

    due_date_was_present = (
        df["_raw_due_date"].notna()
        & df["_raw_due_date"].ne("")
    )


    df["flag_invalid_invoice_date"] = (
        invoice_date_was_present
        & df["invoice_date"].isna()
    ).astype("boolean")


    df["flag_invalid_due_date"] = (
        due_date_was_present
        & df["due_date"].isna()
    ).astype("boolean")


    # --------------------------------------------------------
    # 8.9 Standardize text fields
    # --------------------------------------------------------

    df["region"] = (
        df["region"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    df["region"] = df["region"].replace(
        {
            "Tanger": "Tangier",
            "Marrakech": "Marrakesh",
        }
    )


    df["currency"] = (
        df["currency"]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    df["payment_method"] = (
        df["payment_method"]
        .astype("string")
        .str.strip()
        .str.title()
    )


    df["status"] = (
        df["status"]
        .astype("string")
        .str.strip()
        .str.title()
    )


    df["customer_name"] = (
        df["customer_name"]
        .astype("string")
        .str.strip()
    )


    # --------------------------------------------------------
    # 8.10 Remove exact duplicates
    # --------------------------------------------------------

    rows_before_duplicates = len(df)

    df = (
        df
        .drop_duplicates(
            keep="first"
        )
        .reset_index(drop=True)
    )

    rows_after_duplicates = len(df)

    exact_duplicates_removed = (
        rows_before_duplicates
        - rows_after_duplicates
    )


    # --------------------------------------------------------
    # 8.11 Missing reference-data flags
    # --------------------------------------------------------

    df["flag_missing_customer_id"] = (
        df["customer_id"].isna()
    ).astype("boolean")


    df["flag_missing_region"] = (
        df["region"].isna()
    ).astype("boolean")


    df["flag_missing_payment_method"] = (
        df["payment_method"].isna()
    ).astype("boolean")


    # --------------------------------------------------------
    # 8.12 Financial flags
    # --------------------------------------------------------

    df["flag_negative_invoice"] = (
        df["invoice_amount"] < 0
    ).astype("boolean")


    df.loc[
        df["invoice_amount"].isna(),
        "flag_negative_invoice"
    ] = pd.NA


    df["flag_negative_payment"] = (
        df["payment_amount"] < 0
    ).astype("boolean")


    df["flag_zero_payment"] = (
        df["payment_amount"] == 0
    ).astype("boolean")


    df["flag_payment_exceeds_invoice"] = (
        (
            df["payment_amount"].notna()
            & df["invoice_amount"].notna()
            & (
                df["payment_amount"]
                > df["invoice_amount"]
            )
        )
        .fillna(False)
        .astype("boolean")
    )


    # --------------------------------------------------------
    # 8.13 Date relationship flags
    # --------------------------------------------------------

    df["flag_early_payment"] = (
        (
            df["payment_date"].notna()
            & df["due_date"].notna()
            & (
                df["payment_date"]
                < df["due_date"]
            )
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_payment_before_invoice"] = (
        (
            df["payment_date"].notna()
            & df["invoice_date"].notna()
            & (
                df["payment_date"]
                < df["invoice_date"]
            )
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_due_before_invoice"] = (
        (
            df["due_date"].notna()
            & df["invoice_date"].notna()
            & (
                df["due_date"]
                < df["invoice_date"]
            )
        )
        .fillna(False)
        .astype("boolean")
    )


    # --------------------------------------------------------
    # 8.14 Status / payment-evidence flags
    # --------------------------------------------------------

    df["flag_paid_missing_payment_amount"] = (
        (
            (df["status"] == "Paid")
            & df["payment_amount"].isna()
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_paid_missing_payment_date"] = (
        (
            (df["status"] == "Paid")
            & df["payment_date"].isna()
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_open_with_payment_evidence"] = (
        (
            (df["status"] == "Open")
            & (
                df["payment_amount"].notna()
                | df["payment_date"].notna()
            )
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_payment_without_date"] = (
        (
            df["payment_amount"].notna()
            & df["payment_date"].isna()
        )
        .fillna(False)
        .astype("boolean")
    )


    df["flag_payment_date_without_amount"] = (
        (
            df["payment_date"].notna()
            & df["payment_amount"].isna()
        )
        .fillna(False)
        .astype("boolean")
    )


    # --------------------------------------------------------
    # 8.15 Original status
    # --------------------------------------------------------

    df["original_status"] = df["status"]


    # --------------------------------------------------------
    # 8.16 Derived date fields
    # --------------------------------------------------------

    df["invoice_year"] = (
        df["invoice_date"].dt.year
    )

    df["invoice_month"] = (
        df["invoice_date"].dt.month
    )

    df["invoice_year_month"] = (
        df["invoice_date"]
        .dt.to_period("M")
        .astype("string")
    )


    # --------------------------------------------------------
    # 8.17 Payment timing metrics
    # --------------------------------------------------------

    df["days_to_payment"] = (
        df["payment_date"]
        - df["invoice_date"]
    ).dt.days


    df["days_from_due_date"] = (
        df["payment_date"]
        - df["due_date"]
    ).dt.days


    df["payment_timing"] = pd.Series(
        pd.NA,
        index=df.index,
        dtype="string"
    )


    early_mask = (
        (
            df["payment_date"].notna()
            & df["due_date"].notna()
            & (
                df["payment_date"]
                < df["due_date"]
            )
        )
        .fillna(False)
    )


    on_due_mask = (
        (
            df["payment_date"].notna()
            & df["due_date"].notna()
            & (
                df["payment_date"]
                == df["due_date"]
            )
        )
        .fillna(False)
    )


    late_mask = (
        (
            df["payment_date"].notna()
            & df["due_date"].notna()
            & (
                df["payment_date"]
                > df["due_date"]
            )
        )
        .fillna(False)
    )


    df.loc[
        early_mask,
        "payment_timing"
    ] = "Early"


    df.loc[
        on_due_mask,
        "payment_timing"
    ] = "On Due Date"


    df.loc[
        late_mask,
        "payment_timing"
    ] = "Late"


    # --------------------------------------------------------
    # 8.18 Payment reconciliation
    # --------------------------------------------------------

    invoice = df["invoice_amount"]
    payment = df["payment_amount"]


    reconciliation_conditions = [
        invoice.isna().to_numpy(
            dtype=bool
        ),

        payment.isna().to_numpy(
            dtype=bool
        ),

        (
            payment < 0
        ).fillna(False).to_numpy(
            dtype=bool
        ),

        (
            payment == 0
        ).fillna(False).to_numpy(
            dtype=bool
        ),

        (
            payment > invoice
        ).fillna(False).to_numpy(
            dtype=bool
        ),

        (
            payment == invoice
        ).fillna(False).to_numpy(
            dtype=bool
        ),

        (
            payment < invoice
        ).fillna(False).to_numpy(
            dtype=bool
        ),
    ]


    reconciliation_choices = [
        "Missing Invoice Amount",
        "Missing Payment Amount",
        "Negative Payment",
        "Zero Payment",
        "Payment Exceeds Invoice",
        "Fully Matched",
        "Partial Payment",
    ]


    df["payment_reconciliation"] = np.select(
        reconciliation_conditions,
        reconciliation_choices,
        default="Other"
    )


    # --------------------------------------------------------
    # 8.19 Quality issue count
    # --------------------------------------------------------

    flag_columns = [
        "flag_missing_invoice_amount",
        "flag_invalid_invoice_amount",
        "flag_unusable_invoice_amount",
        "flag_missing_customer_id",
        "flag_missing_region",
        "flag_missing_payment_method",
        "flag_invalid_invoice_date",
        "flag_invalid_due_date",
        "flag_negative_invoice",
        "flag_negative_payment",
        "flag_zero_payment",
        "flag_payment_exceeds_invoice",
        "flag_early_payment",
        "flag_payment_before_invoice",
        "flag_due_before_invoice",
        "flag_paid_missing_payment_amount",
        "flag_paid_missing_payment_date",
        "flag_open_with_payment_evidence",
        "flag_payment_without_date",
        "flag_payment_date_without_amount",
    ]


    for col in flag_columns:
        df[col] = df[col].astype("boolean")


    df["quality_issue_count"] = (
        df[flag_columns]
        .fillna(False)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # 8.20 Business rule status
    # --------------------------------------------------------

    incomplete_mask = (
        df["flag_unusable_invoice_amount"]
        .fillna(False)
    )


    review_mask = (
        df["flag_payment_exceeds_invoice"]
        .fillna(False)

        | df["flag_negative_invoice"]
        .fillna(False)

        | df["flag_negative_payment"]
        .fillna(False)

        | df["flag_zero_payment"]
        .fillna(False)

        | df["flag_paid_missing_payment_amount"]
        .fillna(False)

        | df["flag_paid_missing_payment_date"]
        .fillna(False)

        | df["flag_open_with_payment_evidence"]
        .fillna(False)

        | df["flag_payment_without_date"]
        .fillna(False)

        | df["flag_payment_date_without_amount"]
        .fillna(False)
    )


    expected_mask = (
        (df["status"] == "Open")
        & df["payment_amount"].isna()
        & df["payment_date"].isna()
        & ~incomplete_mask
        & ~review_mask
    )


    df["business_rule_status"] = np.select(
        [
            safe_bool(incomplete_mask),
            safe_bool(review_mask),
            safe_bool(expected_mask),
        ],
        [
            "INCOMPLETE",
            "REVIEW",
            "EXPECTED",
        ],
        default="VALID"
    )


    # --------------------------------------------------------
    # 8.21 Financial analytical eligibility
    #
    # EXACTLY reproduces Phase 4:
    #
    # invoice missing → NOT_USABLE
    # invoice present + payment missing → PARTIAL
    # invoice present + payment present → FULL
    # --------------------------------------------------------

    full_mask = (
        df["invoice_amount"].notna()
        & df["payment_amount"].notna()
    )


    partial_mask = (
        df["invoice_amount"].notna()
        & df["payment_amount"].isna()
    )


    df["financial_analytical_eligibility"] = np.select(
        [
            df["invoice_amount"]
            .isna()
            .to_numpy(dtype=bool),

            safe_bool(partial_mask),

            safe_bool(full_mask),
        ],
        [
            "NOT_USABLE_FOR_AMOUNT_ANALYSIS",
            "PARTIAL",
            "FULL",
        ],
        default="NOT_USABLE_FOR_AMOUNT_ANALYSIS"
    )


    # --------------------------------------------------------
    # 8.22 Business rule reasons
    #
    # EXACTLY reproduces Phase 4 priority/order.
    # --------------------------------------------------------

    reason_conditions = [
        (
            df["flag_missing_invoice_amount"],
            "MISSING_INVOICE_AMOUNT"
        ),

        (
            df["flag_invalid_invoice_amount"],
            "INVALID_INVOICE_AMOUNT"
        ),

        (
            df["flag_missing_customer_id"],
            "MISSING_CUSTOMER_ID"
        ),

        (
            df["flag_missing_region"],
            "MISSING_REGION"
        ),

        (
            df["flag_missing_payment_method"],
            "MISSING_PAYMENT_METHOD"
        ),

        (
            df["flag_negative_invoice"],
            "NEGATIVE_INVOICE"
        ),

        (
            df["flag_negative_payment"],
            "NEGATIVE_PAYMENT"
        ),

        (
            df["flag_zero_payment"],
            "ZERO_PAYMENT"
        ),

        (
            df["flag_payment_exceeds_invoice"],
            "PAYMENT_EXCEEDS_INVOICE"
        ),

        (
            df["flag_early_payment"],
            "EARLY_PAYMENT"
        ),

        (
            df["flag_payment_before_invoice"],
            "PAYMENT_BEFORE_INVOICE"
        ),

        (
            df["flag_due_before_invoice"],
            "DUE_DATE_BEFORE_INVOICE"
        ),

        (
            df["flag_paid_missing_payment_amount"],
            "PAID_MISSING_PAYMENT_AMOUNT"
        ),

        (
            df["flag_paid_missing_payment_date"],
            "PAID_MISSING_PAYMENT_DATE"
        ),

        (
            df["flag_open_with_payment_evidence"],
            "OPEN_WITH_PAYMENT_EVIDENCE"
        ),

        (
            df["flag_payment_without_date"],
            "PAYMENT_WITHOUT_DATE"
        ),

        (
            df["flag_payment_date_without_amount"],
            "PAYMENT_DATE_WITHOUT_AMOUNT"
        ),

        (
            df["flag_invalid_invoice_date"],
            "INVALID_INVOICE_DATE"
        ),

        (
            df["flag_invalid_due_date"],
            "INVALID_DUE_DATE"
        ),
    ]


    reason_parts = []


    for mask, reason in reason_conditions:

        safe_mask = (
            mask
            .fillna(False)
            .to_numpy(dtype=bool)
        )

        reason_parts.append(
            pd.Series(
                np.where(
                    safe_mask,
                    reason,
                    ""
                ),
                index=df.index
            )
        )


    reason_df = pd.concat(
        reason_parts,
        axis=1
    )


    df["business_rule_reason"] = (
        reason_df
        .fillna("")
        .agg(
            lambda row: ";".join(
                value
                for value in row
                if value != ""
            ),
            axis=1
        )
    )


    df["business_rule_reason"] = (
        df["business_rule_reason"]
        .replace(
            "",
            "NO_EXCEPTION"
        )
    )


    # --------------------------------------------------------
    # 8.23 Downstream analytical flags
    # --------------------------------------------------------

    df["is_invoice_amount_usable"] = (
        df["invoice_amount"].notna()
    ).astype("boolean")


    df["is_payment_amount_usable"] = (
        df["payment_amount"].notna()
    ).astype("boolean")


    df["has_payment_evidence"] = (
        df["payment_amount"].notna()
        | df["payment_date"].notna()
    ).astype("boolean")


    # IMPORTANT:
    # This reproduces Phase 4 exactly.
    #
    # Phase 4 definition:
    # Open + usable invoice amount
    #
    # It does NOT require payment_amount/payment_date
    # to be missing.
    df["is_outstanding_receivable_candidate"] = (
        (df["status"] == "Open")
        & df["invoice_amount"].notna()
    ).astype("boolean")


    df["is_payment_performance_candidate"] = (
        (df["status"] == "Paid")
        & df["invoice_amount"].notna()
        & df["payment_date"].notna()
    ).astype("boolean")


    # --------------------------------------------------------
    # 8.24 Quality status
    # --------------------------------------------------------

    quality_critical = (
        df["flag_unusable_invoice_amount"]
        .fillna(False)
    )


    quality_investigate = (
        review_mask
        .fillna(False)
    )


    quality_review = (
        df["flag_missing_customer_id"]
        .fillna(False)

        | df["flag_missing_region"]
        .fillna(False)

        | df["flag_missing_payment_method"]
        .fillna(False)

        | df["flag_invalid_invoice_date"]
        .fillna(False)

        | df["flag_invalid_due_date"]
        .fillna(False)
    )


    df["quality_status"] = np.select(
        [
            safe_bool(quality_critical),
            safe_bool(quality_investigate),
            safe_bool(quality_review),
        ],
        [
            "CRITICAL",
            "INVESTIGATE",
            "REVIEW",
        ],
        default="PASS"
    )


    # --------------------------------------------------------
    # 8.25 Reorder columns
    # --------------------------------------------------------

    core_columns = [
        "invoice_id",
        "customer_id",
        "customer_name",
        "region",
        "supplier_id",
        "product_id",
        "invoice_date",
        "due_date",
        "payment_date",
        "invoice_amount",
        "payment_amount",
        "currency",
        "payment_method",
        "status",
    ]


    temporary_columns = [
        "_raw_invoice_amount",
        "_raw_invoice_date",
        "_raw_due_date",
        "_invoice_amount_is_missing",
        "_invoice_amount_numeric",
    ]


    remaining_columns = [
        col
        for col in df.columns
        if col not in core_columns
        and col not in temporary_columns
    ]


    df = df[
        core_columns
        + remaining_columns
    ]


    # Remove technical temporary columns.
    #
    # The actual raw values have already served their purpose.
    # They are not needed in the final analytical dataset.
    #
    # This also keeps Phase 5 at the same 52-column business
    # schema as Phase 4.

    return (
        df.reset_index(drop=True),
        exact_duplicates_removed
    )


# ============================================================
# 9. INTERNAL VALIDATION
# ============================================================

def validate_pipeline(
    df,
    exact_duplicates_removed
):

    validation_results = []


    def record_check(
        check_name,
        condition,
        details
    ):

        validation_results.append(
            {
                "check": check_name,
                "result":
                    "PASS"
                    if bool(condition)
                    else "FAIL",
                "details": details,
            }
        )


    # --------------------------------------------------------
    # Row / duplicate checks
    # --------------------------------------------------------

    record_check(
        "Row count after duplicate removal",
        len(df) == 12000,
        f"Rows = {len(df):,}"
    )


    record_check(
        "Exactly 110 duplicate rows removed",
        exact_duplicates_removed == 110,
        f"Removed = {exact_duplicates_removed:,}"
    )


    record_check(
        "Invoice IDs are unique",
        df["invoice_id"].is_unique,
        (
            f"Rows = {len(df):,}; "
            f"Unique IDs = "
            f"{df['invoice_id'].nunique():,}"
        )
    )


    record_check(
        "Invoice ID has no missing values",
        df["invoice_id"].notna().all(),
        (
            "Missing invoice IDs = "
            f"{df['invoice_id'].isna().sum():,}"
        )
    )


    record_check(
        "No exact duplicate rows remain",
        not df.duplicated().any(),
        (
            "Duplicate rows = "
            f"{df.duplicated().sum():,}"
        )
    )


    # --------------------------------------------------------
    # Date relationships
    # --------------------------------------------------------

    payment_before_invoice = (
        (
            df["payment_date"].notna()
            & df["invoice_date"].notna()
            & (
                df["payment_date"]
                < df["invoice_date"]
            )
        )
        .sum()
    )


    record_check(
        "No payment before invoice",
        payment_before_invoice == 0,
        f"Violations = {payment_before_invoice:,}"
    )


    due_before_invoice = (
        (
            df["due_date"].notna()
            & df["invoice_date"].notna()
            & (
                df["due_date"]
                < df["invoice_date"]
            )
        )
        .sum()
    )


    record_check(
        "No due date before invoice",
        due_before_invoice == 0,
        f"Violations = {due_before_invoice:,}"
    )


    # --------------------------------------------------------
    # Categories
    # --------------------------------------------------------

    allowed_status = {
        "Open",
        "Paid",
    }


    unexpected_status = sorted(
        set(
            df["status"]
            .dropna()
            .unique()
        )
        - allowed_status
    )


    record_check(
        "Status categories are valid",
        len(unexpected_status) == 0,
        f"Unexpected = {unexpected_status}"
    )


    allowed_currency = {
        "MAD",
        "EUR",
    }


    unexpected_currency = sorted(
        set(
            df["currency"]
            .dropna()
            .unique()
        )
        - allowed_currency
    )


    record_check(
        "Currency categories are valid",
        len(unexpected_currency) == 0,
        f"Unexpected = {unexpected_currency}"
    )


    allowed_regions = {
        "Casablanca",
        "Rabat",
        "Tangier",
        "Marrakesh",
        "Fes",
        "Agadir",
        "Oujda",
    }


    unexpected_regions = sorted(
        set(
            df["region"]
            .dropna()
            .unique()
        )
        - allowed_regions
    )


    record_check(
        "Region categories are standardized",
        len(unexpected_regions) == 0,
        f"Unexpected = {unexpected_regions}"
    )


    # --------------------------------------------------------
    # Invoice amount validation
    # --------------------------------------------------------

    missing_invoice_count = int(
        df["flag_missing_invoice_amount"]
        .sum()
    )


    invalid_invoice_count = int(
        df["flag_invalid_invoice_amount"]
        .sum()
    )


    unusable_invoice_count = int(
        df["flag_unusable_invoice_amount"]
        .sum()
    )


    record_check(
        "Missing invoice amount count is 30",
        missing_invoice_count == 30,
        f"Missing = {missing_invoice_count:,}"
    )


    record_check(
        "Invalid invoice amount count is 80",
        invalid_invoice_count == 80,
        f"Invalid = {invalid_invoice_count:,}"
    )


    record_check(
        "Unusable invoice amount count is 110",
        unusable_invoice_count == 110,
        f"Unusable = {unusable_invoice_count:,}"
    )


    # --------------------------------------------------------
    # Negative invoices
    # --------------------------------------------------------

    negative_invoice_count = int(
        df["flag_negative_invoice"]
        .fillna(False)
        .sum()
    )


    negative_invoice_unknown = int(
        df["flag_negative_invoice"]
        .isna()
        .sum()
    )


    record_check(
        "Negative invoice count is 79",
        negative_invoice_count == 79,
        f"Negative = {negative_invoice_count:,}"
    )


    record_check(
        "Negative invoice is unknown for unusable amounts",
        (
            negative_invoice_unknown == 110
            and
            df.loc[
                df["flag_negative_invoice"].isna(),
                "invoice_amount"
            ].isna().all()
        ),
        (
            "Unknown negative status = "
            f"{negative_invoice_unknown:,}"
        )
    )


    # --------------------------------------------------------
    # Business-rule completeness
    # --------------------------------------------------------

    blank_reason_count = int(
        (
            df["business_rule_reason"].isna()
            | df["business_rule_reason"].eq("")
        )
        .sum()
    )


    record_check(
        "Business rule reason has no blanks",
        blank_reason_count == 0,
        f"Blank reasons = {blank_reason_count:,}"
    )


    blank_status_count = int(
        df["business_rule_status"]
        .isna()
        .sum()
    )


    record_check(
        "Business rule status has no blanks",
        blank_status_count == 0,
        f"Blank statuses = {blank_status_count:,}"
    )


    blank_eligibility_count = int(
        df["financial_analytical_eligibility"]
        .isna()
        .sum()
    )


    record_check(
        "Financial analytical eligibility has no blanks",
        blank_eligibility_count == 0,
        (
            "Blank eligibility = "
            f"{blank_eligibility_count:,}"
        )
    )


    # --------------------------------------------------------
    # Required schema
    # --------------------------------------------------------

    record_check(
        "Final dataset has 52 columns",
        len(df.columns) == 52,
        f"Columns = {len(df.columns):,}"
    )


    validation_df = pd.DataFrame(
        validation_results
    )


    return validation_df


# ============================================================
# 10. NORMALIZATION FOR REGRESSION
# ============================================================

def normalize_for_comparison(
    series,
    column_name
):

    # Dates
    if (
        pd.api.types.is_datetime64_any_dtype(
            series
        )
    ):

        return (
            pd.to_datetime(
                series,
                errors="coerce"
            )
            .dt.strftime("%Y-%m-%d")
            .fillna("<NA>")
        )


    # Numeric columns
    if (
        pd.api.types.is_numeric_dtype(
            series
        )
    ):

        return (
            pd.to_numeric(
                series,
                errors="coerce"
            )
            .round(10)
            .astype("string")
            .fillna("<NA>")
        )


    # Everything else
    return (
        series
        .astype("string")
        .str.strip()
        .fillna("<NA>")
    )


# ============================================================
# 11. REGRESSION TEST
# ============================================================

def regression_test_against_phase4(
    phase5_df,
    phase4_df
):

    phase5 = phase5_df.copy()
    phase4 = phase4_df.copy()


    # --------------------------------------------------------
    # Standardize column names
    # --------------------------------------------------------

    phase5 = standardize_column_names(
        phase5
    )

    phase4 = standardize_column_names(
        phase4
    )


    # --------------------------------------------------------
    # Check row counts
    # --------------------------------------------------------

    regression_results = []
    detail_rows = []


    if len(phase5) != len(phase4):

        regression_results.append(
            {
                "column": "__ROW_COUNT__",
                "result": "FAIL",
                "differences": (
                    f"Phase 5 = {len(phase5):,}; "
                    f"Phase 4 = {len(phase4):,}"
                ),
            }
        )

        return (
            pd.DataFrame(regression_results),
            pd.DataFrame(detail_rows)
        )


    # --------------------------------------------------------
    # Normalize invoice IDs FOR ALIGNMENT ONLY
    # --------------------------------------------------------

    phase5["_regression_invoice_id"] = (
        phase5["invoice_id"]
        .astype("string")
        .str.strip()
    )


    phase4["_regression_invoice_id"] = (
        phase4["invoice_id"]
        .astype("string")
        .str.strip()
    )


    # --------------------------------------------------------
    # Compare ID sets
    # --------------------------------------------------------

    phase5_ids = set(
        phase5["_regression_invoice_id"]
        .dropna()
    )


    phase4_ids = set(
        phase4["_regression_invoice_id"]
        .dropna()
    )


    missing_from_phase5 = (
        phase4_ids - phase5_ids
    )

    missing_from_phase4 = (
        phase5_ids - phase4_ids
    )


    if (
        missing_from_phase5
        or missing_from_phase4
    ):

        regression_results.append(
            {
                "column": "invoice_id",
                "result": "FAIL",
                "differences": (
                    f"Missing from Phase 5: "
                    f"{len(missing_from_phase5):,}; "
                    f"Missing from Phase 4: "
                    f"{len(missing_from_phase4):,}"
                ),
            }
        )

        return (
            pd.DataFrame(regression_results),
            pd.DataFrame(detail_rows)
        )


    # --------------------------------------------------------
    # Sort both datasets by invoice ID.
    #
    # This eliminates false failures caused only by
    # different row ordering.
    # --------------------------------------------------------

    phase5 = (
        phase5
        .sort_values(
            "_regression_invoice_id"
        )
        .reset_index(drop=True)
    )


    phase4 = (
        phase4
        .sort_values(
            "_regression_invoice_id"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Common columns only
    # --------------------------------------------------------

    common_columns = [
        col
        for col in phase5.columns
        if (
            col in phase4.columns
            and col != "_regression_invoice_id"
        )
    ]


    # --------------------------------------------------------
    # Compare every common column
    # --------------------------------------------------------

    for column in common_columns:

        p5_normalized = normalize_for_comparison(
            phase5[column],
            column
        )

        p4_normalized = normalize_for_comparison(
            phase4[column],
            column
        )


        difference_mask = (
            p5_normalized
            != p4_normalized
        )


        difference_count = int(
            difference_mask.sum()
        )


        if difference_count == 0:

            regression_results.append(
                {
                    "column": column,
                    "result": "PASS",
                    "differences": 0,
                }
            )

            continue


        regression_results.append(
            {
                "column": column,
                "result": "FAIL",
                "differences": difference_count,
            }
        )


        # Store row-level differences
        differing_indices = (
            np.where(
                difference_mask.to_numpy()
            )[0]
        )


        for idx in differing_indices:

            detail_rows.append(
                {
                    "invoice_id":
                        phase5.loc[
                            idx,
                            "_regression_invoice_id"
                        ],

                    "column":
                        column,

                    "phase5_value":
                        phase5.loc[
                            idx,
                            column
                        ],

                    "phase4_value":
                        phase4.loc[
                            idx,
                            column
                        ],
                }
            )


    regression_df = pd.DataFrame(
        regression_results
    )


    regression_detail_df = pd.DataFrame(
        detail_rows
    )


    return (
        regression_df,
        regression_detail_df
    )


# ============================================================
# 12. EXCEPTION REPORT
# ============================================================

def create_exception_report(df):

    exception_mask = (
        df["business_rule_status"]
        .isin(
            [
                "INCOMPLETE",
                "REVIEW",
            ]
        )
    )


    return (
        df.loc[
            exception_mask
        ]
        .copy()
    )


# ============================================================
# 13. QUALITY REPORT
# ============================================================

def create_quality_report(df):

    quality_report = pd.DataFrame(
        [
            {
                "quality_status": status,
                "row_count": int(
                    (
                        df["quality_status"]
                        == status
                    ).sum()
                ),
            }
            for status in [
                "PASS",
                "REVIEW",
                "INVESTIGATE",
                "CRITICAL",
            ]
        ]
    )


    return quality_report


# ============================================================
# 14. SHA-256
# ============================================================

def calculate_sha256(path):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:

        for chunk in iter(
            lambda:
                file.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)


    return sha256.hexdigest()


# ============================================================
# 15. MAIN PIPELINE
# ============================================================

def run_pipeline():

    print("=" * 70)
    print("PHASE 5 — REPEATABLE ETL PIPELINE")
    print("=" * 70)

    print(
        f"Run ID       : {RUN_ID}"
    )

    print(
        f"Raw input    : {RAW_PATH}"
    )

    print(
        f"Phase 4 ref  : {PHASE4_PATH}"
    )

    print(
        f"Output dir   : {OUTPUT_DIR}"
    )


    # ========================================================
    # STEP 1 — EXTRACT
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 1 — EXTRACT")
    print("-" * 70)


    raw_df = extract_data()


    print(
        f"Raw rows     : {len(raw_df):,}"
    )

    print(
        f"Raw columns  : {len(raw_df.columns):,}"
    )


    # ========================================================
    # STEP 2 — TRANSFORM
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 2 — TRANSFORM / CLEAN")
    print("-" * 70)


    clean_df, exact_duplicates_removed = (
        transform_data(
            raw_df
        )
    )


    print(
        f"Rows before dedup        : "
        f"{len(raw_df):,}"
    )

    print(
        f"Exact duplicates removed : "
        f"{exact_duplicates_removed:,}"
    )

    print(
        f"Rows after dedup         : "
        f"{len(clean_df):,}"
    )

    print(
        f"Final columns            : "
        f"{len(clean_df.columns):,}"
    )


    # ========================================================
    # STEP 3 — INTERNAL VALIDATION
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 3 — PIPELINE VALIDATION")
    print("-" * 70)


    validation_df = validate_pipeline(
        clean_df,
        exact_duplicates_removed
    )


    print(
        validation_df.to_string(
            index=False
        )
    )


    validation_df.to_csv(
        VALIDATION_REPORT_PATH,
        index=False
    )


    validation_failed = (
        validation_df["result"]
        == "FAIL"
    ).any()


    if validation_failed:

        print()
        print(
            "WARNING: One or more internal "
            "validation checks FAILED."
        )

        print(
            "The Phase 5 output will still be "
            "saved so the failure can be inspected."
        )

    else:

        print()
        print(
            "ALL INTERNAL VALIDATION CHECKS PASS."
        )


    # ========================================================
    # STEP 4 — SAVE OUTPUT
    #
    # IMPORTANT:
    # Save BEFORE regression.
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 4 — SAVE PHASE 5 OUTPUT")
    print("-" * 70)


    clean_df.to_csv(
        OUTPUT_PATH,
        index=False
    )


    print(
        f"Output saved successfully:\n"
        f"{OUTPUT_PATH}"
    )


    # ========================================================
    # STEP 5 — LOAD PHASE 4 REFERENCE
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 5 — LOAD PHASE 4 REFERENCE")
    print("-" * 70)


    if not PHASE4_PATH.exists():

        raise FileNotFoundError(
            f"Phase 4 reference file not found:\n"
            f"{PHASE4_PATH}"
        )


    phase4_df = pd.read_csv(
        PHASE4_PATH
    )


    print(
        f"Phase 4 rows    : "
        f"{len(phase4_df):,}"
    )

    print(
        f"Phase 4 columns : "
        f"{len(phase4_df.columns):,}"
    )


    # ========================================================
    # STEP 6 — REGRESSION
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 6 — REGRESSION TEST AGAINST PHASE 4")
    print("-" * 70)


    (
        regression_df,
        regression_detail_df
    ) = regression_test_against_phase4(
        clean_df,
        phase4_df
    )


    regression_df.to_csv(
        REGRESSION_REPORT_PATH,
        index=False
    )


    regression_detail_df.to_csv(
        REGRESSION_DETAIL_PATH,
        index=False
    )


    passed_count = int(
        (
            regression_df["result"]
            == "PASS"
        ).sum()
    )


    failed_count = int(
        (
            regression_df["result"]
            == "FAIL"
        ).sum()
    )


    print()
    print(
        f"Columns compared : "
        f"{len(regression_df):,}"
    )

    print(
        f"Columns passed   : "
        f"{passed_count:,}"
    )

    print(
        f"Columns failed   : "
        f"{failed_count:,}"
    )


    if failed_count == 0:

        print()
        print(
            "REGRESSION TEST: PASS"
        )

    else:

        print()
        print(
            "REGRESSION TEST: DIFFERENCES FOUND"
        )

        print()
        print(
            regression_df[
                regression_df["result"]
                == "FAIL"
            ]
            .to_string(index=False)
        )

        print()
        print(
            "Detailed row-level differences:"
        )

        print(
            REGRESSION_DETAIL_PATH
        )


    # ========================================================
    # STEP 7 — EXCEPTION REPORT
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 7 — EXCEPTION REPORT")
    print("-" * 70)


    exceptions_df = (
        create_exception_report(
            clean_df
        )
    )


    exceptions_df.to_csv(
        EXCEPTION_REPORT_PATH,
        index=False
    )


    print(
        f"Exception rows : "
        f"{len(exceptions_df):,}"
    )


    # ========================================================
    # STEP 8 — QUALITY REPORT
    # ========================================================

    print()
    print("-" * 70)
    print("STEP 8 — QUALITY REPORT")
    print("-" * 70)


    quality_df = create_quality_report(
        clean_df
    )


    quality_df.to_csv(
        QUALITY_REPORT_PATH,
        index=False
    )


    print(
        quality_df.to_string(
            index=False
        )
    )


    # ========================================================
    # STEP 9 — RUN LOG
    # ========================================================

    run_log_df = pd.DataFrame(
        [
            {
                "run_id": RUN_ID,
                "run_timestamp": RUN_TIMESTAMP,
                "raw_path": str(RAW_PATH),
                "phase4_reference": str(PHASE4_PATH),
                "output_path": str(OUTPUT_PATH),
                "raw_rows": len(raw_df),
                "raw_columns": len(raw_df.columns),
                "duplicates_removed":
                    exact_duplicates_removed,
                "final_rows": len(clean_df),
                "final_columns": len(clean_df.columns),
                "validation_failed":
                    bool(validation_failed),
                "regression_passed":
                    failed_count == 0,
                "regression_failed_columns":
                    failed_count,
            }
        ]
    )


    run_log_df.to_csv(
        RUN_LOG_PATH,
        index=False
    )


    # ========================================================
    # STEP 10 — SHA-256
    # ========================================================

    output_hash = calculate_sha256(
        OUTPUT_PATH
    )


    sha256_df = pd.DataFrame(
        [
            {
                "run_id": RUN_ID,
                "file": str(OUTPUT_PATH),
                "sha256": output_hash,
            }
        ]
    )


    sha256_df.to_csv(
        SHA256_REPORT_PATH,
        index=False
    )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("PHASE 5 — COMPLETE")
    print("=" * 70)

    print()
    print(
        f"Rows                    : "
        f"{len(clean_df):,}"
    )

    print(
        f"Columns                 : "
        f"{len(clean_df.columns):,}"
    )

    print(
        f"Duplicates removed      : "
        f"{exact_duplicates_removed:,}"
    )

    print()
    print("Invoice amount classification:")

    print(
        f"  Missing               : "
        f"{int(clean_df['flag_missing_invoice_amount'].sum()):,}"
    )

    print(
        f"  Invalid               : "
        f"{int(clean_df['flag_invalid_invoice_amount'].sum()):,}"
    )

    print(
        f"  Unusable              : "
        f"{int(clean_df['flag_unusable_invoice_amount'].sum()):,}"
    )

    print(
        f"  Valid numeric         : "
        f"{int(clean_df['invoice_amount'].notna().sum()):,}"
    )

    print()
    print("Business rule status:")

    print(
        clean_df[
            "business_rule_status"
        ]
        .value_counts()
        .to_string()
    )

    print()
    print("Financial analytical eligibility:")

    print(
        clean_df[
            "financial_analytical_eligibility"
        ]
        .value_counts()
        .to_string()
    )

    print()
    print("Regression:")

    print(
        f"  Passed columns        : "
        f"{passed_count:,}"
    )

    print(
        f"  Failed columns        : "
        f"{failed_count:,}"
    )

    print()
    print("Output files:")

    print(
        f"  {OUTPUT_PATH}"
    )

    print(
        f"  {VALIDATION_REPORT_PATH}"
    )

    print(
        f"  {REGRESSION_REPORT_PATH}"
    )

    print(
        f"  {REGRESSION_DETAIL_PATH}"
    )

    print(
        f"  {EXCEPTION_REPORT_PATH}"
    )

    print(
        f"  {QUALITY_REPORT_PATH}"
    )

    print(
        f"  {RUN_LOG_PATH}"
    )

    print(
        f"  {SHA256_REPORT_PATH}"
    )

    print()
    print(
        f"SHA-256: {output_hash}"
    )

    print()
    print(
        "PHASE 5 PIPELINE FINISHED."
    )

    return clean_df


# ============================================================
# 16. RUN PHASE 5
# ============================================================

clean_df = run_pipeline()

PHASE 5 — REPEATABLE ETL PIPELINE
Run ID       : 20260927_174450
Raw input    : C:\Users\hp\Downloads\Working_capital_dirty .csv
Phase 4 ref  : C:\Users\hp\Downloads\Working_capital_validated.csv
Output dir   : C:\Users\hp\Downloads\Working_Capital_ETL

----------------------------------------------------------------------
STEP 1 — EXTRACT
----------------------------------------------------------------------
Raw rows     : 12,110
Raw columns  : 14

----------------------------------------------------------------------
STEP 2 — TRANSFORM / CLEAN
----------------------------------------------------------------------
Rows before dedup        : 12,110
Exact duplicates removed : 110
Rows after dedup         : 12,000
Final columns            : 52

----------------------------------------------------------------------
STEP 3 — PIPELINE VALIDATION
----------------------------------------------------------------------
                                           check result                     